# STEP 0: RAW DATA

In [ ]:
# New ismports for section 
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import calendar

In [ ]:

# Show all columns in full when printing a DataFrame
pd.set_option('display.max_columns', None)

# Optional: also show full width if columns are wide
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)


In [ ]:
url = "bananas-250818.csv"
raw_data = pd.read_csv(url)

In [ ]:
# Set raw response for all proceeding functions
raw_response_var = 'Price'

In [ ]:
# Set global response for all proceeding functions
global_response_var = 'Price'

In [ ]:
# Check info 
raw_data.info()

In [ ]:
# Look at sample
raw_data.head(5)

In [ ]:
raw_data.describe()

In [ ]:
# Missing % by column
print("Missing Records % by Variable\n",(raw_data.isnull().sum()/len(raw_data))*100)

In [ ]:
# Missing % by row
print("%Rows Missing Records\n",(raw_data.isnull().any(axis=1).mean() * 100))

In [ ]:
# Return any duplicates
raw_data[raw_data.duplicated(keep=False)]

In [ ]:
# Checking missingness graphically
sns.heatmap(raw_data.isnull(),yticklabels=False,cbar=False,cmap='viridis')

# STEP 1: RINSED DATA

In [ ]:
# Ensure some checkpoints for analysis
rinsed_data = raw_data.copy()

In [ ]:
rinsed_data.info()

### Formatting 
- Rinse data before imputing, getting rid of all anomanlies besides missing values 

In [ ]:
# drop: PassengerId, Name, Cabin, Ticket
# object to cat: Sex, Embarked
# int to cat:  Survived, Pclass, SibSp

In [ ]:
# Drop anything over 30% missing, too unique or not unique enough 
rinsed_data.drop('Units',axis=1, inplace=True) # missingness over 30%
# rinsed_data.drop('PassengerId',axis=1, inplace=True) # ID variables, too unique
# rinsed_data.drop('Name',axis=1, inplace=True) # ID variables, too unique
# rinsed_data.drop('Ticket',axis=1, inplace=True) # Single categories, not unique enough
rinsed_data = rinsed_data[rinsed_data["Origin"]!='acp_bananas']
rinsed_data = rinsed_data[rinsed_data["Origin"]!='dollar_bananas']
rinsed_data = rinsed_data[rinsed_data["Origin"]!='all_bananas']

In [ ]:
# Convert to datetime (day first since it's DD/MM/YYYY)
rinsed_data["Date"] = pd.to_datetime(rinsed_data["Date"], format="%d/%m/%Y")

# Create new columns
rinsed_data["Month_Name"] = rinsed_data["Date"].dt.month_name()
rinsed_data["Month_Num"] = rinsed_data["Date"].dt.month
rinsed_data["Year"] = rinsed_data["Date"].dt.year

In [ ]:
rinsed_data.info()

In [ ]:
rinsed_data.sample()

In [ ]:
# Change all int64 that should category to category
rinsed_data['Month_Name'] = rinsed_data['Month_Name'].astype('category')
rinsed_data['Origin'] = rinsed_data['Origin'].astype('category')


In [ ]:
rinsed_data.info()

In [ ]:

rinsed_data.drop('Date',axis=1, inplace=True) # not predictor 


### Prior to Imputation
- Note what is skew and what is imbalanced and how it will affect further analysis

In [ ]:
# check the proportions of each group 
for var in (rinsed_data.select_dtypes(include=['object', 'category'])).columns:
    print(rinsed_data[var].value_counts(normalize=True))

In [ ]:
# Check univariate distribution of numeric
sns.pairplot(rinsed_data)

In [ ]:
rinsed_data.head(5)

In [ ]:
rinsed_data.info()

# STEP 3: BASIC EDA

In [ ]:
imputed_data = rinsed_data.copy()

In [ ]:
imputed_data.info()

In [ ]:
# New imports for section

# Association analysis for Categorical Variables
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency
from scipy import stats

In [ ]:
stats.stats.spearmanr(imputed_data[['Price','Year']])

In [ ]:

ax = sns.lineplot(data=imputed_data, x = 'Year',y='Price')
ax.set(xlabel='Year', ylabel='Price (Pounds/kg)')
ax.set_ylim(0, None)
plt.title("Average Banana Price by Year")
plt.tight_layout()
plt.savefig('Banana Price by Year.png')
plt.show()

In [ ]:
# Heatmap on Imputed Data without categories 

ax = sns.lineplot(data=imputed_data, x = 'Month_Num',y='Price')
ax.set_ylim(0, None)
ax.set_xticks(range(1, 13))  # Ensure all 12 months show
ax.set(xlabel='Months', ylabel='Price (Pounds/kg)')
ax.set_xticklabels(calendar.month_abbr[1:])
plt.title("Banana Price by Month")
plt.tight_layout()
plt.savefig('Banana Price by Month.png')
plt.show()

In [ ]:
# ZOOM IN ON NUMERIC 

ax= sns.boxplot(data=imputed_data.sort_values(by="Month_Num"), hue='Month_Name', x = global_response_var, palette="icefire", gap=.2, notch=True, whis=(0, 100))
ax.set(xlabel='Price (Pounds/kg)')
# Show the plots
plt.tight_layout()
plt.savefig('Boxplot of Banana Price by Month')
plt.show()


In [ ]:
imputed_data[imputed_data['Price']>2]

In [ ]:
# Plot for Loan Status == 0 on the second axis
ax = sns.kdeplot(data=imputed_data, x=global_response_var)
ax.set(xlabel='Price (Pounds/kg)')
plt.title("Distributions of Banana Price by Month")
plt.tight_layout()
plt.savefig('Distribution of Banana Price.png')
plt.show()

In [ ]:
# Plot for Loan Status == 0 on the second axis
ax = sns.kdeplot(data=imputed_data, x=global_response_var, hue='Month_Name',palette="icefire")
ax.set(xlabel='Price (Pounds/kg)')
plt.title("Distributions of Banana Price by Month")
plt.tight_layout()
plt.savefig('Distributions of Banana Price by Month.png')
plt.show()


In [ ]:
# Get unique Origin categories
origins = imputed_data["Origin"].unique()

# Find midpoint
midpoint = len(origins) // 2

# Split into two groups
origins_1 = origins[:midpoint]
origins_2 = origins[midpoint:]

# Create two subsets
subset_1 = imputed_data[imputed_data["Origin"].isin(origins_1)]
subset_2 = imputed_data[imputed_data["Origin"].isin(origins_2)]


In [ ]:
# Box for subset_1
ax = sns.boxplot(
    data=subset_1.sort_values(by="Origin"),
    hue="Origin",
    x=global_response_var,
    palette="icefire",
    gap=0.2,
    notch=True,
    whis=(0, 100),
    hue_order=subset_1["Origin"].unique()  # restrict legend to only these
)

ax.set(xlabel="Price (Pounds/kg)", title = "Boxplot of Banana Price by Origin")
plt.legend(title="Origin")
ax.set_xlim(0, 2.5)
plt.minorticks_on()
plt.tight_layout()
plt.savefig("Boxplot of Banana Price by Origin1.png")
plt.show()


In [ ]:
# Box for subset_1
ax = sns.boxplot(
    data=subset_2.sort_values(by="Origin"),
    hue="Origin",
    x=global_response_var,
    palette="icefire",
    gap=0.2,
    notch=True,
    whis=(0, 100),
    hue_order=subset_2["Origin"].unique()  # restrict legend to only these
)

ax.set(xlabel="Price (Pounds/kg)", title = "Boxplot of Banana Price by Origin")
plt.legend(title="Origin")
ax.set_xlim(0, 2.5)
plt.minorticks_on()
plt.tight_layout()
plt.savefig("Boxplot of Banana Price by Origin2.png")
plt.show()


# STEP 4: MODELLING 

### Setting up modelling 

In [ ]:
last_5 = imputed_data[imputed_data["Year"]>=2020].copy()
last_5.drop('Month_Num',axis=1, inplace=True) # missingness over 30%




last_5['Year'] = last_5['Year'].astype('category')

# Origin_acp_bananas: -0.06138492115926378
# Origin_dollar_bananas: 0.008842811414304467
last_5.info()

In [ ]:
imputed_data[imputed_data["Origin"]=='acp_bananas']

In [ ]:
last_5.info()

In [ ]:
last_5["Origin"].nunique()

In [ ]:
len(last_5)

In [ ]:
# New imports for section

# Encoding to dummies 
from pandas.api.types import CategoricalDtype

# standard Linear regression section
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from sklearn import metrics

# standard data split
from sklearn.model_selection import train_test_split

# k fold cross validation 
from sklearn.model_selection import cross_val_score

# bootstrap 
from sklearn.metrics import accuracy_score

# to mute the wanrnigs for cells like logistic regression convergence and deprecated functions
import warnings

# Polynomial 
from sklearn.preprocessing import PolynomialFeatures

# GAMs
from pygam import LinearGAM, s

# Splines
from scipy.interpolate import UnivariateSpline

# GAM Formula 
from patsy import dmatrix

# Naive Decision Tree 
from sklearn.tree import DecisionTreeRegressor

# Random Forest 
from sklearn.ensemble import RandomForestRegressor

# ADA Boost 
from sklearn.ensemble import AdaBoostRegressor

# Tuning
from sklearn.model_selection import GridSearchCV



In [ ]:
imputed_data[imputed_data["Origin"] != "eu"]["Origin"].unique()

In [ ]:
test_df = last_5.copy() 

# Set 'eu' as reference by putting it first
test_df["Origin"] = pd.Categorical(test_df["Origin"], categories=["eu"] + 
                                           [level for level in test_df["Origin"].unique() if level != "eu"])


# Create dummies, dropping reference, and add prefix with variable name
origin_dummies = pd.get_dummies(test_df["Origin"], drop_first=True).add_prefix("Origin_")

# Concatenate on X set 
test_df = pd.concat([test_df, origin_dummies], axis=1)

np.shape(origin_dummies)

In [ ]:
origin_dummies.info()

In [ ]:
test_df.info()

In [ ]:
test_df.sample()

In [ ]:
# PREP DATA FOR MODELLING

def manual_dummy(df, response_var=global_response_var):
    
    # Make copy of df that was passed
    dummy_encoded_X = df.drop(response_var, axis=1)
    # Get the response in its own df 
    y = df[response_var]
    
    # Get dummies from X
    dummy_encoded_X.columns
        
    # Set 'eu' as reference by putting it first
    dummy_encoded_X["Origin"] = pd.Categorical(dummy_encoded_X["Origin"], categories=["eu"] + 
                                               [level for level in dummy_encoded_X["Origin"].unique() if level != "eu"])
    dummy_encoded_X["Month_Name"] = pd.Categorical(dummy_encoded_X["Month_Name"], categories=["January"] + 
                                               [level for level in dummy_encoded_X["Month_Name"].unique() if level != "January"])
    dummy_encoded_X["Year"] = pd.Categorical(dummy_encoded_X["Year"], categories=[2020] + 
                                               [level for level in dummy_encoded_X["Year"].unique() if level != 2020])
    
    # Create dummies, dropping reference, and add prefix with variable name
    origin_dummies = pd.get_dummies(dummy_encoded_X["Origin"], drop_first=True).add_prefix("Origin_")
    month_dummies = pd.get_dummies(dummy_encoded_X["Month_Name"], drop_first=True).add_prefix("Month_")
    year_dummies = pd.get_dummies(dummy_encoded_X["Year"], drop_first=True).add_prefix("Year_")
    
    
    # Concatenate on X set 
    dummy_encoded_X = pd.concat([dummy_encoded_X, origin_dummies,month_dummies,year_dummies], axis=1)
    
    # Drop the original categorical column
    dummy_encoded_X.drop(columns=['Origin','Month_Name','Year'], axis=1, inplace=True)
    

    # Encode any new dummy variables to 1s and 0s 
    dummy_encoded_X[dummy_encoded_X.select_dtypes(include=['bool']).columns] = dummy_encoded_X.select_dtypes(include=['bool']).astype(int)
   
    # Ensure new names do not cause problems
    dummy_encoded_X.columns = dummy_encoded_X.columns.astype(str)

    return dummy_encoded_X, y

X,y = manual_dummy(last_5)


X.info()

In [ ]:
# GENERAL STRUCTURE FOR TRAIN/TEST SPLIT 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1) # 0.25 x 0.8 = 0.2



### Linear Regression

In [ ]:
# LOGISTIC REGRESSION


# train model
lm = LinearRegression()
lm_results = lm.fit(X_train,y_train)


# make preds and evaluate
lm_valid_preds = lm_results.predict(X_val)
lm_train_preds = lm_results.predict(X_train)
lm_test_preds = lm_results.predict(X_test)

print("Performance Report: Train Set")
print('MAE:', metrics.mean_absolute_error(y_train, lm_train_preds))
print('MSE:', metrics.mean_squared_error(y_train, lm_train_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_train, lm_train_preds)))
print('R-squared:', metrics.r2_score(y_train, lm_train_preds))
print()
print("Performance Report: Validation Set")
print('MAE:', metrics.mean_absolute_error(y_val, lm_valid_preds))
print('MSE:', metrics.mean_squared_error(y_val, lm_valid_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_val, lm_valid_preds)))
print('R-squared:', metrics.r2_score(y_val, lm_valid_preds))


In [ ]:
plt.scatter(y_test,lm_valid_preds)
plt.title('Linear Regression - Banana Prices')
# plt.xlabel('Median Income')
plt.ylabel('Banana Price (Pounds/kg)')
plt.show()

### K-Fold CV

In [ ]:
# GENERAL PURPOSE K-FOLD

# k fold
def k_fold(model,X,y,k):
    # Get predicted labels using k-fold cross-validation
    mae_scores = cross_val_score(lm, X_train, y_train, scoring='neg_median_absolute_error', cv=k)
    mse_scores = cross_val_score(lm, X_train, y_train, scoring='neg_mean_squared_error', cv=k)
    rmse_scores = cross_val_score(lm, X_train, y_train, scoring='neg_root_mean_squared_error', cv=k)
    r2_scores = cross_val_score(lm, X_train, y_train, scoring='r2', cv=k)

    print("Performance Report: K-Fold CV")
    print('MAE:', (-1)*np.mean(mae_scores))
    print('MSE:', (-1)*np.mean(mse_scores))
    print('RMSE:', (-1)*np.mean(rmse_scores))
    print('R-squared:',np.mean(r2_scores))


In [ ]:
# DO K-FOLD

k = 5

k_fold(lm,X_train,y_train,k)

### Naive Decision Tree

In [ ]:
# NAIVE DECISION TREE

param_grid = {
    "max_depth": [None, 5, 10, 20, 50,100,200],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [None, "sqrt", "log2"]
}
 
grid = GridSearchCV(DecisionTreeRegressor(random_state=42),
                    param_grid,
                    cv=5,
                    scoring="neg_mean_squared_error",
                    n_jobs=-1)

grid.fit(X_train, y_train)

tree_regressor = grid.best_estimator_

print("Params:\n",tree_regressor.get_params())

# make preds and evaluate
tree_valid_preds = tree_regressor.predict(X_val)
tree_train_preds = tree_regressor.predict(X_train)
tree_test_preds = tree_regressor.predict(X_test)

print("Performance Report: Train Set")
print('MAE:', metrics.mean_absolute_error(y_train, tree_train_preds))
print('MSE:', metrics.mean_squared_error(y_train, tree_train_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_train, tree_train_preds)))
print('R-squared:', metrics.r2_score(y_train, tree_train_preds))
print()
print("Performance Report: Validation Set")
print('MAE:', metrics.mean_absolute_error(y_val, tree_valid_preds))
print('MSE:', metrics.mean_squared_error(y_val, tree_valid_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_val, tree_valid_preds)))
print('R-squared:', metrics.r2_score(y_val, tree_valid_preds))

### Random Forest 

In [ ]:
# RANDOM FOREST 



param_grid = {
    "n_estimators": [50, 100,200],
    "max_depth": [None, 10, 20, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

grid = GridSearchCV(RandomForestRegressor(random_state=42),
                    param_grid,
                    cv=5,
                    scoring="neg_mean_squared_error",
                    n_jobs=-1)

grid.fit(X_train, y_train)
random_forest_regressor = grid.best_estimator_


# make preds and evaluate
forest_train_preds = random_forest_regressor.predict(X_train)
forest_valid_preds = random_forest_regressor.predict(X_val)
forest_test_preds = random_forest_regressor.predict(X_test)

print("Params:\n",random_forest_regressor.get_params())

print("Performance Report: Train Set")
print('MAE:', metrics.mean_absolute_error(y_train, forest_train_preds))
print('MSE:', metrics.mean_squared_error(y_train, forest_train_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_train, forest_train_preds)))
print('R-squared:', metrics.r2_score(y_train, forest_train_preds))
print()
print("Performance Report: Validation Set")
print('MAE:', metrics.mean_absolute_error(y_val, forest_valid_preds))
print('MSE:', metrics.mean_squared_error(y_val, forest_valid_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_val, forest_valid_preds)))
print('R-squared:', metrics.r2_score(y_val, forest_valid_preds))


### AdaBoost

In [ ]:
# ADABOOST 
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "estimator": [
        DecisionTreeRegressor(max_depth=d, random_state=42) for d in [2, 3, 5, 10]
    ]
}

grid = GridSearchCV(AdaBoostRegressor(random_state=42),
                    param_grid,
                    cv=5,
                    scoring="neg_mean_squared_error",
                    n_jobs=-1)

grid.fit(X_train, y_train)
adaboost_regressor = grid.best_estimator_


# make preds and evaluate
adaboost_train_preds = adaboost_regressor.predict(X_train)
adaboost_valid_preds = adaboost_regressor.predict(X_val)
adaboost_test_preds = adaboost_regressor.predict(X_test)


print("Performance Report: Train Set")
print('MAE:', metrics.mean_absolute_error(y_train, adaboost_train_preds))
print('MSE:', metrics.mean_squared_error(y_train, adaboost_train_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_train, adaboost_train_preds)))
print('R-squared:', metrics.r2_score(y_train, adaboost_train_preds))
print()
print("Performance Report: Validation Set")
print('MAE:', metrics.mean_absolute_error(y_val, adaboost_valid_preds))
print('MSE:', metrics.mean_squared_error(y_val, adaboost_valid_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_val, adaboost_valid_preds)))
print('R-squared:', metrics.r2_score(y_val, adaboost_valid_preds))

### Model Evaluation

In [ ]:
### Final Model Evaluation
# X_val, y_val


In [ ]:
# metrics for comparisons 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# plotting logistic
from numpy import argmax, sqrt
from sklearn.metrics import roc_curve
from matplotlib import pyplot
from sklearn.metrics import precision_recall_curve

# plot trees
from sklearn.tree import plot_tree
from dtreeviz import model
from xgboost import plot_tree


In [ ]:
# MAKE DATAFRAME

# Model names and predictions
model_names = ['Linear Regression', 'Decision Tree', 'Random Forest', 'AdaBoost']
predictions = [lm_test_preds, tree_test_preds, forest_test_preds, adaboost_test_preds]


print('MAE:', metrics.mean_absolute_error(y_val, adaboost_valid_preds))
print('MSE:', metrics.mean_squared_error(y_val, adaboost_valid_preds))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_val, adaboost_valid_preds)))
print('R-squared:', metrics.r2_score(y_val, adaboost_valid_preds))

# Compute metrics
data = []
for name, y_pred in zip(model_names, predictions):
    data.append({
        'Model': name,
        'MAE': round(metrics.mean_absolute_error(y_test, y_pred),4),
        'MSE': round(metrics.mean_squared_error(y_test, y_pred),4),
        'RMSE': round(np.sqrt(metrics.mean_squared_error(y_test, y_pred)),4),
        'R-squared': round(metrics.r2_score(y_test, y_pred),4),
    })

# Create DataFrame
perform_df = pd.DataFrame(data)
print(perform_df)


#### Evaluation Metrics


Here are three common evaluation metrics for regression problems:

**Mean Absolute Error** (MAE) is the mean of the absolute value of the errors:

$$\frac 1n\sum_{i=1}^n|y_i-\hat{y}_i|$$

**Mean Squared Error** (MSE) is the mean of the squared errors:

$$\frac 1n\sum_{i=1}^n(y_i-\hat{y}_i)^2$$

**Root Mean Squared Error** (RMSE) is the square root of the mean of the squared errors:

$$\sqrt{\frac 1n\sum_{i=1}^n(y_i-\hat{y}_i)^2}$$

Comparing these metrics:

- **MAE** the average error.
- **MSE** MSE "punishes" larger errors.
- **RMSE** RMSE is interpretable in the "y" units.

All of these are **loss functions** and we want to minimize them.

In [ ]:
# Melt the dataframe to long format
df_melted = perform_df.sort_values(by=['R-squared','RMSE'], ascending=True).melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=df_melted,palette='Blues')
plt.title('Comparison of Model Performance')
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.tight_layout()
plt.savefig('Comparison of Model Performance.png')
plt.show()


In [ ]:
# Melt the dataframe to long format
df_melted = perform_df.sort_values(by=['R-squared','RMSE'], ascending=True).melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Model', y='Score', hue='Metric', data=df_melted, palette='Blues')

# Add value labels on top of each bar
for p in ax.patches:
    height = p.get_height()
    ax.text(
        x=p.get_x() + p.get_width() / 2,  # X-coordinate: center of the bar
        y=height + 0.01,                 # Y-coordinate: just above the bar
        s=f'{height:.4f}',               # Text to display (formatted)
        ha='center',                     # Center horizontally
        va='bottom',                     # Align vertically to the bottom of the text
        fontsize=9
    )

plt.title('Comparison of Model Performance')
plt.ylim(0, 1.1)  # Slightly higher than 1 to make space for labels
plt.legend(title='Metric')
plt.tight_layout()
plt.savefig('Annotated Comparison of Model Performance.png')
plt.show()


### Linear Regression Variable Importance

In [ ]:
# print the coefficients
print("Intercept:")
print(round(lm_results.intercept_,4))

print("Coefficients:")
for feature, coef in zip(X.columns, lm_results.coef_):
    print(f"{feature}: {np.round(coef,4)}")


In [ ]:
# Build the coefficient dictionary dynamically (excluding intercept)
coef_dict = {feature: np.round(coef, 4) 
             for feature, coef in zip(X.columns, lm_results.coef_)}

# Convert to DataFrame for plotting
coef_df = pd.DataFrame({
    'Feature': coef_dict.keys(),
    'Coefficient': coef_dict.values()
})

# Determine color by sign
coef_df['Color'] = np.where(coef_df['Coefficient'] >= 0, 'green', 'red')

# Absolute magnitude for sorting
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=True)

# Plot horizontal bar chart
plt.figure(figsize=(8, 12))
plt.barh(coef_df['Feature'], coef_df['Abs'], color=coef_df['Color'])
plt.legend(['Positive', 'Negative'], loc='best', facecolor='white')
plt.legend(handles=[plt.Rectangle((0,0),1,1,color='green'),
                    plt.Rectangle((0,0),1,1,color='red')],
           labels=['Positive','Negative'], loc='best')
plt.xlabel('Absolute Coefficient Magnitude')
plt.ylabel('Features')
plt.title('Linear Regression Coefficients Magnitude')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('Linear Regression Coefficients Magnitude.png')
plt.show()


### Random Forest Variable Importance

In [ ]:

# Get feature importances
importances = random_forest_regressor.feature_importances_

# Create a series with feature names
feature_names = X.columns if isinstance(X, pd.DataFrame) else [f"Feature {i}" for i in range(X.shape[1])]
# get feature importance
importance_series = pd.Series(importances, index=feature_names)

# Sort feature importances
importance_series = importance_series.sort_values(ascending=True)

# Plot 
plt.figure(figsize=(10, 6))
importance_series.plot(kind='barh',colormap ='viridis' )
plt.title('Random Forest Feature Importance')
plt.ylabel('Features')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('Random Forest Feature Importance.png')
plt.show()

